# Data Analysis for Titanic Dataset on Kaggler

## 1. Setup and Dataset loading

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

In [2]:
train = pd.read_csv("../data/train.csv")
test = pd.read_csv("../data/test.csv")

## 2. Exploratory Data Analysis

In [3]:
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB


In [38]:
#train[train.isna().any(axis=1)].info()
train[train.notnull().all(axis=1)].info()

<class 'pandas.DataFrame'>
Index: 183 entries, 1 to 889
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  183 non-null    int64  
 1   Survived     183 non-null    int64  
 2   Pclass       183 non-null    int64  
 3   Name         183 non-null    str    
 4   Sex          183 non-null    str    
 5   Age          183 non-null    float64
 6   SibSp        183 non-null    int64  
 7   Parch        183 non-null    int64  
 8   Ticket       183 non-null    str    
 9   Fare         183 non-null    float64
 10  Cabin        183 non-null    str    
 11  Embarked     183 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 18.6 KB


The dataset have three columns with at least one NaN value, leaving only 183 entries with no NaN values, instead of using the regular method for handling NaN values, we'll use ML models that can handle these type of cases. We also have to transform any values into NaN if necessary

In [5]:
train.nunique()

PassengerId    891
Survived         2
Pclass           3
Name           891
Sex              2
Age             88
SibSp            7
Parch            7
Ticket         681
Fare           248
Cabin          147
Embarked         3
dtype: int64

In [6]:
train.describe()

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


In [7]:
train.head(10)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
5,6,0,3,"Moran, Mr. James",male,NaN,0,0,330877,8.4583,NaN,Q
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463,51.8625,E46,S
7,8,0,3,"Palsson, Master. Gosta Leonard",male,2.0,3,1,349909,21.0750,NaN,S
8,9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27.0,0,2,347742,11.1333,NaN,S
9,10,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",female,14.0,1,0,237736,30.0708,NaN,C


We can see the column Embarked could be splited into three different columns *"Embarked_S", "Embarked_C",* and *"Embarked_Q"* for better perfomance during the training of the model. Also, if necessary, we could tokenize the passanger's names

### 2.1 Univariate anaylsis

In [151]:
categorical_cols = ['Pclass', 'Sex', 'Embarked', 'Survived']
numerical_cols = ['Age', 'Fare']

In [154]:
for col in categorical_cols:
    px.pie(train, 
           names=col, 
           template="seaborn", 
           title=f'Passengers by {col}',
           ).show()

By just looking at the charts we can guess Sex is the most important variable to predict if a passenger is going to survive or not

In [155]:
for col in numerical_cols:
    fig = px.histogram(train, x=col)
    fig.show()
    print(f"{col} stats: \n", train[col].describe(), f"Median: {train[col].median()}\n")

Age stats: 
 count    714.000000
mean      29.699118
std       14.526497
min        0.420000
25%       20.125000
50%       28.000000
75%       38.000000
max       80.000000
Name: Age, dtype: float64 Median: 28.0



Fare stats: 
 count    891.000000
mean      32.204208
std       49.693429
min        0.000000
25%        7.910400
50%       14.454200
75%       31.000000
max      512.329200
Name: Fare, dtype: float64 Median: 14.4542



We can see there's a signfiicant outlier within the Fare column, with three passengers paying a fare of >500 pounds, we'll have to find out if this outlier must be removed before training the model to avoid biases

### 2.2 Bivariate analysis

In [160]:
for col in categorical_cols:
    if col == "Survived":
        continue

    survival_rate = train["Survived"].groupby(train[col]).mean() * 100
    survival_rate = survival_rate.reset_index()

    fig = px.bar(survival_rate,
                 x=col,
                 y="Survived",
                 template="seaborn",
                 title=f"Survival Rate by {col}")
    fig.show()
    print(survival_rate)

   Pclass   Survived
0       1  62.962963
1       2  47.282609
2       3  24.236253


      Sex   Survived
0  female  74.203822
1    male  18.890815


  Embarked   Survived
0        C  55.357143
1        Q  38.961039
2        S  33.695652
